# Generate MCAR reference files

Runs `pyampute` on deterministic datasets generated with `ehrdata.dt.ehrdata_blobs`
and writes the expected p-values to the reference files that the MCAR tests read.


In [1]:
import json
from pathlib import Path

import ehrdata as ed
import numpy as np
import pandas as pd
from pyampute.exploration.mcar_statistical_tests import MCARTest

In [2]:
# Parameters must match the constants in tests/preprocessing/test_quality_control.py
_BLOB_KWARGS = {"n_centers": 1, "cluster_std": 1.0, "base_timepoints": 1}

_SCENARIOS_LITTLE = {
    "mcar_small": {"n_obs": 100, "n_vars": 10, "missing_rate": 0.10, "seed": 42},
    "mar_small": {"n_obs": 100, "n_vars": 10, "missing_pct": 0.10, "seed": 42},
    "mcar_medium_high_missing": {"n_obs": 900, "n_vars": 50, "missing_rate": 0.50, "seed": 7},
}
_SCENARIO_TTEST = {"n_obs": 200, "n_vars": 8, "missing_pct": 0.20, "seed": 99}

In [3]:
def _make_mcar_edata(*, n_obs, n_vars, missing_rate, seed):
    return ed.dt.ehrdata_blobs(
        n_observations=n_obs, n_variables=n_vars, missing_values=missing_rate, random_state=seed, **_BLOB_KWARGS
    )


def _make_mar_edata(*, n_obs, n_vars, missing_pct, seed):
    edata = ed.dt.ehrdata_blobs(
        n_observations=n_obs, n_variables=n_vars, missing_values=0.0, random_state=seed, **_BLOB_KWARGS
    )
    X = np.asarray(edata.X, dtype=float).copy()
    X[X[:, 0] < np.percentile(X[:, 0], missing_pct * 100), -1] = np.nan
    edata.X = X
    return edata


def _build_little_scenario(name):
    cfg = _SCENARIOS_LITTLE[name]
    return _make_mcar_edata(**cfg) if "missing_rate" in cfg else _make_mar_edata(**cfg)


def _to_df(edata):
    return pd.DataFrame(np.asarray(edata.X, dtype=float), columns=list(edata.var_names))

In [4]:
out_dir = Path("../../tests/data/preprocessing/mcar_refs")
out_dir.mkdir(parents=True, exist_ok=True)

little = MCARTest(method="little")
little_expected = {name: float(little(_to_df(_build_little_scenario(name)))) for name in _SCENARIOS_LITTLE}

with (out_dir / "little_expected.json").open("w", encoding="utf-8") as f:
    json.dump(little_expected, f, indent=2, sort_keys=True)

print(little_expected)

{'mcar_small': 0.8127459507579607, 'mar_small': 2.484164678229206e-05, 'mcar_medium_high_missing': 0.001045380246672778}


In [5]:
ttest = MCARTest(method="ttest")
ttest_result = ttest(_to_df(_make_mar_edata(**_SCENARIO_TTEST)))
ttest_result.to_csv(out_dir / "ttest_mar_expected.csv")

ttest_result

/home/andreas/.local/share/hatch/env/virtual/ehrapy/Kw9Jk1rX/docs/lib64/python3.14/site-packages/pyampute/exploration/mcar_statistical_tests.py:146: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  mcar_matrix.loc[var, tvar] = ttest_ind(


,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7
feature_0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
feature_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
feature_2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
feature_3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
feature_4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
feature_5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
feature_6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
feature_7,2.936954e-34,0.34972,0.966874,0.941181,0.961084,0.578553,0.433705,NaN
